# Capstone 2: Customer Churn Prediction

### Business Context

A subscription business (telecom, streaming, SaaS) is losing customers. You must build a model that predicts which customers are likely to churn and identify the main drivers so the retention team can act — precisely the scenario Volumes 05 and 06 built toward throughout the entire course.

### Step 1: Data and Exploratory Analysis

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
n = 2000
df = pd.DataFrame({
    "tenure_months": np.random.randint(1, 72, n),
    "monthly_charges": np.round(np.random.uniform(25, 150, n), 2),
    "contract": np.random.choice(["Month-to-month", "One year", "Two year"], n, p=[0.55, 0.25, 0.2]),
    "internet_service": np.random.choice(["DSL", "Fiber", "None"], n, p=[0.35, 0.45, 0.2]),
    "tech_support": np.random.choice(["Yes", "No"], n, p=[0.3, 0.7]),
    "senior_citizen": np.random.choice([0, 1], n, p=[0.85, 0.15]),
    "num_tickets": np.random.poisson(1.5, n)
})
df["total_charges"] = (df["tenure_months"] * df["monthly_charges"] * np.random.uniform(0.9, 1.1, n)).round(2)

logit = (-2.0 + 0.025*df["monthly_charges"] - 0.04*df["tenure_months"]
         + 0.8*(df["contract"] == "Month-to-month").astype(int)
         + 0.4*(df["tech_support"] == "No").astype(int)
         + 0.3*df["num_tickets"] + 0.5*df["senior_citizen"])
df["churn"] = (np.random.rand(n) < 1/(1+np.exp(-logit))).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(data=df, x="contract", hue="churn", ax=axes[0]); axes[0].set_title("Churn by Contract")
sns.boxplot(data=df, x="churn", y="tenure_months", ax=axes[1]); axes[1].set_title("Tenure vs Churn")
sns.boxplot(data=df, x="churn", y="monthly_charges", ax=axes[2]); axes[2].set_title("Monthly Charges vs Churn")
plt.tight_layout()
plt.show()

Running exploratory visualizations *before* modeling (exactly this three-panel comparison) is not a skippable formality — it's how you build the intuition to sanity-check the model's results later. If the model's top feature importance came back as something these EDA charts show no visible relationship with, that would be a signal to investigate a possible bug or data leakage issue before trusting the model at all.

### Step 2: Feature Preparation and Modeling

In [ ]:
df_ml = pd.get_dummies(df, columns=["contract", "internet_service", "tech_support"], drop_first=True)
X = df_ml.drop(columns=["churn", "total_charges"])     # total_charges dropped: highly correlated with tenure (redundant)
y = df_ml["churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
num_cols = ["tenure_months", "monthly_charges", "num_tickets"]
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])          # fit on train, transform both — Volume 05's golden rule

logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)

for name, model in [("Logistic Regression", logreg), ("Decision Tree", tree)]:
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    print(f"\n=== {name} ===")
    print("Accuracy:", round(accuracy_score(y_test, pred), 3))
    print("ROC-AUC :", round(roc_auc_score(y_test, proba), 3))
    print(confusion_matrix(y_test, pred))

💡 **Dropping `total_charges` is a deliberate, important feature-engineering decision worth pausing on:** `total_charges` is almost entirely a mathematical function of `tenure_months × monthly_charges` (with a little noise) — including it alongside both of its own components adds redundant, correlated information (a mild form of **multicollinearity**) without adding real new signal, and can make model coefficients harder to interpret cleanly. Recognizing and removing this kind of derived redundancy is a genuinely important, easily-overlooked step of real feature preparation.

### Step 3: Interpretation and Business Recommendations

In [ ]:
coef = pd.Series(logreg.coef_[0], index=X.columns).sort_values()
print("Logistic Regression Coefficients (impact on churn log-odds):\n", coef.round(3))
print("\nDecision Tree Feature Importances:\n",
      pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False).round(3))

```
Business Recommendations (example, based on typical results from this exact setup):

1. Month-to-month contracts are the strongest churn driver -> incentivize longer
   contracts with a modest discount for annual/two-year commitments.
2. High monthly charges combined with low tenure identifies high-risk new customers
   -> offer loyalty credits during the first 6 months specifically.
3. Customers with many support tickets churn more often -> improve first-call
   resolution and add proactive outreach after a customer's 2nd+ ticket.
4. Deploy the model (Volume 05's joblib pattern) to score the entire customer base
   weekly and feed a ranked "Save" campaign list directly to the retention team.
```

Recommendation 4 is worth calling out specifically: a model that produces a one-time report is useful, but a model that's **deployed to score new/existing customers on an ongoing basis** and feeds directly into an operational workflow (a weekly retention campaign list) is where the real, sustained business value of this entire course's toolkit is actually realized. This is precisely the gap Volume 10's Streamlit and Pydantic chapters, and Volume 05's `joblib` persistence, exist to close — turning a one-off analysis into a running system.

---